# 05 · PyTorch Transformer / BEV Query 训练基线

当前主流的 BEV、检测、tracking 和 planning 模型大量使用 attention，但“知道 attention 这个词”不等于能训练和评测一个时序/多模态模型。本 notebook 用一个小型 PyTorch Transformer 处理 camera/LiDAR token，学习一个 BEV query 的分类任务。

核心公式为

\[
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V.
\]

学习目标：

- 用 `nn.TransformerEncoder` 实现 token mixing；
- 用一个 learned BEV query 汇聚多模态 token；
- 训练、验证、数据扰动和指标报告全部可复现；
- 比较 LiDAR dropout 对模型性能和推理 latency 的影响。

这里不依赖 Hugging Face `transformers`：我们学习的是架构和训练接口，不是加载预训练语言模型。`transformers` 会在 VLM/VLA 前沿分支中作为可选依赖出现。


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from ipywidgets import interact, FloatSlider

torch.set_num_threads(1)
torch.manual_seed(23)
np.random.seed(23)
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
device = torch.device("cpu")
print("torch:", torch.__version__, "device:", device)


## Part A — 构造带有模态交互的 token 数据

每个样本有 6 个 camera token 和 6 个 LiDAR token。标签依赖于两种模态的统计量以及一个交互项，因此只看单一模态不能稳定完成任务。真实 BEV 模型会把 token 换成 image feature、voxel feature、map feature 或 temporal memory。


In [ ]:
def make_token_dataset(n=900, seed=23, d_in=12):
    rng = np.random.default_rng(seed)
    camera = rng.normal(size=(n, 6, d_in)).astype(np.float32)
    lidar = rng.normal(size=(n, 6, d_in)).astype(np.float32)
    camera_signal = camera[:, :, :4].mean(axis=(1, 2))
    lidar_signal = lidar[:, :, :4].mean(axis=(1, 2))
    interaction = camera_signal * lidar_signal
    score = 1.00 * camera_signal + 1.30 * lidar_signal + 0.50 * interaction
    bins = np.quantile(score, [1 / 3, 2 / 3])
    label = np.digitize(score, bins).astype(np.int64)
    tokens = np.concatenate([camera, lidar], axis=1)
    modality = np.concatenate([np.zeros(6, dtype=np.int64), np.ones(6, dtype=np.int64)])
    return torch.tensor(tokens), torch.tensor(label), torch.tensor(modality)


tokens, labels, modality = make_token_dataset()
split = int(0.8 * len(tokens))
train_x, val_x = tokens[:split], tokens[split:]
train_y, val_y = labels[:split], labels[split:]
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=64, shuffle=True)
print("train:", tuple(train_x.shape), "val:", tuple(val_x.shape), "class counts:", torch.bincount(labels).tolist())


In [ ]:
class TinyBEVQueryModel(nn.Module):
    def __init__(self, d_in=12, d_model=48, nhead=4, layers=2, n_classes=3):
        super().__init__()
        self.input_proj = nn.Linear(d_in, d_model)
        self.modality_embedding = nn.Embedding(2, d_model)
        self.query = nn.Parameter(torch.zeros(1, 1, d_model))
        self.position = nn.Parameter(torch.zeros(1, 13, d_model))
        nn.init.normal_(self.query, std=0.02)
        nn.init.normal_(self.position, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4 * d_model,
            dropout=0.0, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, n_classes))

    def forward(self, x, modality_ids):
        h = self.input_proj(x) + self.modality_embedding(modality_ids)
        query = self.query.expand(x.size(0), -1, -1)
        h = torch.cat([query, h], dim=1) + self.position[:, : x.size(1) + 1]
        return self.head(self.encoder(h)[:, 0])


def macro_f1(y_true, y_pred, n_classes=3):
    scores = []
    for cls in range(n_classes):
        tp = ((y_true == cls) & (y_pred == cls)).sum()
        fp = ((y_true != cls) & (y_pred == cls)).sum()
        fn = ((y_true == cls) & (y_pred != cls)).sum()
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        scores.append(2 * precision * recall / max(precision + recall, 1e-8))
    return float(np.mean(scores))


def evaluate(model, x, y, dropout=0.0, noise_std=0.0, seed=91):
    local = torch.Generator().manual_seed(seed)
    perturbed = x.clone()
    if dropout > 0:
        mask = torch.rand(perturbed[:, 6:].shape, generator=local) < dropout
        perturbed[:, 6:] = perturbed[:, 6:].masked_fill(mask, 0.0)
    if noise_std > 0:
        noise = torch.randn(perturbed.shape, generator=local) * noise_std
        perturbed = perturbed + noise
    with torch.no_grad():
        logits = model(perturbed.to(device), modality.to(device))
    pred = logits.argmax(dim=1).cpu()
    accuracy = float((pred == y).float().mean())
    return {"accuracy": accuracy, "macro_f1": macro_f1(y.numpy(), pred.numpy()), "pred": pred}


def train_model(epochs=42):
    model = TinyBEVQueryModel().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_accuracy": []}
    for _ in range(epochs):
        model.train()
        losses = []
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_x, modality)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            losses.append(float(loss))
        model.eval()
        history["train_loss"].append(float(np.mean(losses)))
        history["val_accuracy"].append(evaluate(model, val_x, val_y)["accuracy"])
    return model, history


model, history = train_model()
print(evaluate(model, val_x, val_y))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"])
axes[0].set_title("training loss")
axes[0].set_xlabel("epoch")
axes[1].plot(history["val_accuracy"], color="#d97706")
axes[1].set_title("validation accuracy")
axes[1].set_xlabel("epoch")
plt.show()


def show_robustness(lidar_dropout=0.0, noise_std=0.0):
    model.eval()
    result = evaluate(model, val_x, val_y, dropout=lidar_dropout, noise_std=noise_std)
    print({k: round(v, 4) for k, v in result.items() if k != "pred"})


interact(
    show_robustness,
    lidar_dropout=FloatSlider(value=0.0, min=0.0, max=1.0, step=0.1),
    noise_std=FloatSlider(value=0.0, min=0.0, max=0.8, step=0.05),
)


In [ ]:
def benchmark(model, sample, repeats=80):
    model.eval()
    sample = sample[:1]
    times = []
    with torch.no_grad():
        for _ in range(10):
            _ = model(sample, modality)
        for _ in range(repeats):
            start = time.perf_counter()
            _ = model(sample, modality)
            times.append((time.perf_counter() - start) * 1000)
    return {"p50_ms": np.percentile(times, 50), "p95_ms": np.percentile(times, 95), "p99_ms": np.percentile(times, 99)}


print("batch=1 latency:", {k: round(v, 3) for k, v in benchmark(model, val_x).items()})
print("parameters:", sum(p.numel() for p in model.parameters()))


### 练习与依赖边界

1. 把 learned query 改成两个 query，分别输出 road occupancy 和 interaction risk；
2. 增加 causal temporal mask，把 3 个历史时刻作为 token；
3. 比较 `nhead=1/2/4/8` 的精度与 latency；
4. 用 `torch.profiler` 记录 CPU kernel 和 memory；
5. 安装 `requirements-frontier.txt` 后，只用 `transformers` 加载一个公开 encoder，写一个 adapter 把它的 hidden states 接到本 notebook 的 BEV query head。

关键区分：`torch` 是训练自定义 AD 模型的基础；Hugging Face `transformers` 是预训练 Transformer/VLM 的生态入口，不是所有 BEV、tracking 或 planning notebook 的必需依赖。
